# Pipeline di Preprocessing: Riconoscimento delle Emozioni da Segnali EEG

Questo notebook documenta l'intero flusso di pre-elaborazione dei dati EEG (Elettroencefalogramma) per un task di classificazione delle emozioni. La pipeline segue rigorosamente le best practice della letteratura di dominio e si ispira fortemente al protocollo **SEED-V**.

L'obiettivo di questo script è trasformare i segnali grezzi, registrati durante le sessioni sperimentali, in dataset curati, ricchi di feature informative e pronti per l'addestramento di modelli predittivi.

Per ottimizzare l'uso della memoria (RAM) gestendo tensori di grandi dimensioni, le operazioni pesanti di I/O sono state implementate tramite salvataggi intermedi in formato **HDF5**. 

Il processo si articola in quattro macro-fasi:

1. **Estrazione e Segmentazione Dati (Raw to HDF5):** Lettura dei file `.cnt` originali (62 canali, 1000 Hz) e segmentazione del segnale continuo in finestre temporali di 1.5 secondi coincidenti con la stimolazione visiva. Ad ogni finestra vengono associati i relativi metadati: ID Paziente, ID Sessione, Emozione Target (Label) e intensità percepita (Score).
2. **Filtraggio per Confidenza Emotiva:** Per garantire che il modello apprenda da pattern neurali dove l'emozione target è stata effettivamente suscitata, il dataset viene epurato di tutti i segmenti associati a un'autovalutazione (score) bassa, mantenendo solo i campioni con un'intensità percepita di grado medio-alto (Score ≥ 3).
3. **Estrazione delle Feature (Frequency Domain):** Ogni finestra temporale subisce un filtraggio del segnale (Bandpass 1-50 Hz e Notch 50 Hz) e il riposizionamento tramite *Common Average Reference*. Successivamente, vengono estratte 580 feature per epoca suddivise in cinque bande di frequenza (Delta, Theta, Alpha, Beta, Gamma). Nello specifico vengono usate 3 metriche:
    * **DE (Differential Entropy):** Per misurare la complessità del segnale nei singoli elettrodi.
    * **DASM (Difference Asymmetry)** e **RASM (Rational Asymmetry):** Per catturare le asimmetrie cerebrali tra l'emisfero destro e sinistro, forti indicatori per la valenza emotiva.
4. **Data Splitting:** Mescolamento casuale e suddivisione del dataset finale in tre insiemi (Train al 70%, Validation al 15%, Test al 15%), mantenendo la stratificazione delle etichette emotive per assicurare distribuzioni bilanciate in ogni split. I dati vengono esportati in file CSV indipendenti pronti per la fase di addestramento.

### Inizializzazione  

In [ ]:
# 1. Librerie Standard
import gc
import os

# 2. Librerie di Terze Parti (Scientific Computing & I/O)
import h5py
import mne
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt

# 3. Machine Learning & Deep Learning (sottogruppo di Terze Parti)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

# 1. Estrazione e Segmentazione Dati (Raw to HDF5)

## File intero segmentato

Nella prima parte del processo viene generato il dataset.

Si parte da 9 file `.cnt`, ognuno contenente registrazioni EEG della durata di circa 3000 secondi. Poiché le registrazioni originarie comprendono 66 canali, abbiamo escluso 4 elettrodi non rilevanti, mantenendone solo 62.

Ogni file rappresenta i segnali EEG di un determinato utente in una specifica sessione (il setup prevede 3 utenti per 3 sessioni ciascuno).

Durante ogni sessione, agli utenti vengono mostrati 15 video scelti per suscitare specifiche emozioni. Tra un video e l'altro, ai partecipanti viene dato il tempo di assegnare uno `score` (da 1 a 5) che rappresenta l'intensità con cui hanno percepito quell'emozione.

In fase di estrazione, abbiamo selezionato esclusivamente i segmenti temporali in cui i segnali coincidevano con la visione dei video. Questi segmenti sono stati suddivisi in finestre non sovrapposte da 1.5 secondi. Dato che la frequenza di campionamento è di 1000 Hz, per ogni finestra temporale otteniamo una matrice dove 62 sensori registrano 1500 campioni ciascuno.

A ogni finestra estratta vengono associati dei metadati: l'etichetta dell'emozione target (label), lo score di autovalutazione, il `session_ID` e il `person_ID` (quest'ultimi utili per eventuali test di validazione incrociata tra pazienti).

In conclusione, ogni record del dataset finale in input alla rete è costituito da una matrice di dimensioni **62 × 1500**.

In [ ]:
# ==========================================
# 1. CONFIGURAZIONE PARAMETRI E COSTANTI
# ==========================================
SFREQ = 1000 # Frequenza di campionamento (Hz)
WINDOW_SEC = 1.5 # Lunghezza della finestra temporale in secondi
WINDOW_SIZE = int(WINDOW_SEC * SFREQ) # Risultato: 1500 campioni per ogni segmento
USELESS_CH = ['M1', 'M2', 'VEO', 'HEO'] # Canali non EEG da scartare per evitare rumore elettrooculografico

# ==========================================
# 2. DIZIONARI DEI METADATI SPERIMENTALI
# ==========================================
# Marcatori temporali (inizio e fine in secondi) per i 15 video di ogni sessione
TIMESTAMPS = {
    1: {'start': [30, 132, 287, 555, 773, 982, 1271, 1628, 1730, 2025, 2227, 2435, 2667, 2932, 3204],
        'end':   [102, 228, 524, 742, 920, 1240, 1568, 1697, 1994, 2166, 2401, 2607, 2901, 3172, 3359]},
    2: {'start': [30, 299, 548, 646, 836, 1000, 1091, 1392, 1657, 1809, 1966, 2186, 2333, 2490, 2741],
        'end':   [267, 488, 614, 773, 967, 1059, 1331, 1622, 1777, 1908, 2153, 2302, 2428, 2709, 2817]},
    3: {'start': [30, 353, 478, 674, 825, 908, 1200, 1346, 1451, 1711, 2055, 2307, 2457, 2726, 2888],
        'end':   [321, 418, 643, 764, 877, 1147, 1284, 1418, 1679, 1996, 2275, 2425, 2664, 2857, 3066]}
}

# NUOVA MAPPATURA ETICHETTE (Codifica: 0=Disgust, 1=Fear, 2=Sad, 3=Neutral, 4=Happy)
LABELS_MAP = {
    1: [4, 1, 3, 2, 0] * 3,
    2: [2, 1, 3, 0, 4] + [4, 0, 3, 2, 1] + [3, 4, 1, 2, 0],
    3: [2, 1, 3, 0, 4] + [4, 0, 3, 2, 1] + [3, 4, 1, 2, 0]
}

# TRASCRIZIONE DEGLI SCORE DI AUTOVALUTAZIONE (0-5)
# Struttura: SCORES_MAP[Paziente][Sessione] = [Lista di 15 score]
SCORES_MAP = {
    1: { 
        1: [2, 4, 5, 2, 1, 4, 5, 4, 3, 2, 4, 4, 5, 5, 2],
        2: [3, 2, 5, 3, 4, 3, 5, 4, 4, 3, 5, 2, 5, 5, 2],
        3: [3, 2, 5, 5, 2, 5, 3, 4, 2, 5, 5, 3, 5, 4, 5]
    },
    2: { 
        1: [3, 3, 4, 4, 4, 3, 5, 5, 4, 5, 4, 4, 4, 5, 5],
        2: [4, 2, 5, 4, 4, 4, 3, 5, 5, 4, 4, 5, 5, 4, 1],
        3: [4, 4, 5, 5, 3, 5, 2, 5, 4, 5, 5, 5, 5, 5, 1]
    },
    3: { 
        1: [4, 3, 4, 3, 4, 5, 5, 5, 5, 4, 5, 4, 5, 5, 4],
        2: [4, 3, 5, 3, 4, 4, 5, 4, 5, 4, 5, 5, 5, 5, 4],
        3: [5, 4, 5, 5, 4, 4, 4, 5, 5, 5, 4, 5, 5, 5, 5]
    }
}

FILE_LIST = [
    "1_1_20180804.cnt", "1_2_20180810.cnt", "1_3_20180808.cnt",
    "2_1_20180416.cnt", "2_2_20180419.cnt", "2_3_20180425.cnt",
    "3_1_20180414.cnt", "3_2_20180419.cnt", "3_3_20180424.cnt"
]

# percorso del dataset grezzo
DB_PATH = "../data/processed/eeg_dataset.h5"
DB_PATH_FILTERED = "../data/processed/eeg_dataset_filtered.h5"


In [ ]:
# ==========================================
# 3. CREAZIONE ARCHIVIO HDF5 E PIPELINE DI ESTRAZIONE
# ==========================================
h5_filename = DB_PATH
with h5py.File(h5_filename, 'w') as hf:
    # Inizializzazione dei tensori su disco con dimensione estendibile.
    # y_ds ora ha shape (0, 4) per ospitare lo Score.
    X_ds = hf.create_dataset('X', shape=(0, 62, 1500), maxshape=(None, 62, 1500), dtype='float32', chunks=True)
    y_ds = hf.create_dataset('y', shape=(0, 4), maxshape=(None, 4), dtype='int8')
    
    current_idx = 0 # Puntatore per tracciare la riga di scrittura sul disco
    
    for file_name in FILE_LIST:
        
        path_file = "../data/raw/"+file_name 
       
        if not os.path.exists(path_file):
            continue # Salta il ciclo se il file .cnt manca nella directory
            
        print(f"Estrazione e salvataggio in corso: {path_file}")
        
        # Parsificazione del nome file per dedurre Paziente e Sessione
        parts = file_name.split('_')
        p_id = int(parts[0])
        s_id = int(parts[1])
        
        # Apertura file in modalità "Lazy" (preload=False) per non caricare 50 minuti di EEG in RAM
        raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)
        raw.drop_channels(USELESS_CH, on_missing='ignore')
        
        # Estrazione metadati operativi per il file corrente
        starts, ends = TIMESTAMPS[s_id]['start'], TIMESTAMPS[s_id]['end']
        labels = LABELS_MAP[s_id]
        scores = SCORES_MAP[p_id][s_id] # Estrazione della lista dei 15 score specifici
        
        file_segments = []
        file_meta = []
        
        # Iterazione sui 15 stimoli video
        for i in range(15):
            v_start = starts[i] * SFREQ
            v_end = ends[i] * SFREQ
            
            # MNE estrae dal disco solo la porzione esatta di segnale compresa tra v_start e v_end
            video_data = raw.get_data(start=v_start, stop=v_end).astype(np.float32)
            
            # Calcolo del numero di segmenti interi da 1.5s ricavabili da questo video
            n_windows = video_data.shape[1] // WINDOW_SIZE
            
            for w in range(n_windows):
                w_start = w * WINDOW_SIZE
                w_end = w_start + WINDOW_SIZE
                
                # Taglio del segmento e accodamento alla lista temporanea
                file_segments.append(video_data[:, w_start:w_end])
                
                # Inserimento dei metadati completi: Paziente, Sessione, Emozione Target, Autovalutazione
                file_meta.append([p_id, s_id, labels[i], scores[i]])
                
        # --- FASE DI SCRITTURA SU DISCO ---
        n_new_samples = len(file_segments)
        
        # Espansione della struttura del file H5
        X_ds.resize((current_idx + n_new_samples, 62, 1500))
        y_ds.resize((current_idx + n_new_samples, 4))
        
        # Scrittura effettiva dei blocchi NumPy sul disco rigido
        X_ds[current_idx : current_idx + n_new_samples] = np.array(file_segments, dtype=np.float32)
        y_ds[current_idx : current_idx + n_new_samples] = np.array(file_meta, dtype=np.int8)
        
        # Aggiornamento puntatore per il file successivo
        current_idx += n_new_samples
        
        # --- PULIZIA DELLA RAM ---
        # Rimozione esplicita delle variabili pesanti allocate in questo ciclo
        del raw, file_segments, file_meta, video_data
        gc.collect() # Invocazione forzata del Garbage Collector

print(f"\nGenerazione del dataset HDF5 completata: {h5_filename}")
print(f"Campioni totali estratti: {current_idx}")
print("Struttura metadati (y): [Patient_ID, Session_ID, Label, Score]")

Estrazione e salvataggio in corso: ../data/raw/1_1_20180804.cnt
Estrazione e salvataggio in corso: ../data/raw/1_2_20180810.cnt
Estrazione e salvataggio in corso: ../data/raw/1_3_20180808.cnt
Estrazione e salvataggio in corso: ../data/raw/2_1_20180416.cnt


C:\Users\lucag\AppData\Local\Temp\ipykernel_28024\2952353676.py:83: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)


Estrazione e salvataggio in corso: ../data/raw/2_2_20180419.cnt


C:\Users\lucag\AppData\Local\Temp\ipykernel_28024\2952353676.py:83: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)


Estrazione e salvataggio in corso: ../data/raw/2_3_20180425.cnt


...

Estrazione e salvataggio in corso: ../data/raw/3_2_20180419.cnt


C:\Users\lucag\AppData\Local\Temp\ipykernel_28024\2952353676.py:83: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)


Estrazione e salvataggio in corso: ../data/raw/3_3_20180424.cnt


C:\Users\lucag\AppData\Local\Temp\ipykernel_28024\2952353676.py:83: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(path_file, preload=False, verbose=False)



Generazione del dataset HDF5 completata: ../data/processed/eeg_dataset.h5
Campioni totali estratti: 14691
Struttura metadati (y): [Patient_ID, Session_ID, Label, Score]


## Check Coerenza

In questa sezione eseguiamo una serie di controlli di integrità (sanity check) sul dataset appena generato e salvato in formato HDF5. Questo passaggio è una "best practice" fondamentale per assicurarci che la pipeline di estrazione abbia funzionato correttamente prima di procedere con le operazioni successive, evitando di propagare eventuali errori.

Nello specifico, i test eseguiti verificano:

* **Struttura e Dimensioni:** Viene confermata la presenza dei due tensori principali (`X` per i segnali EEG e `y` per i metadati) e ne viene verificata la *shape* attesa (es. 14691 campioni, 62 canali, 1500 timestep) insieme al tipo di dato (`float32`, ideale per ottimizzare l'uso della memoria mantenendo la precisione decimale necessaria).
* **Integrità del Segnale:** Il tensore dei segnali viene scansionato iterativamente a blocchi (per evitare di saturare la RAM) alla ricerca di eventuali valori mancanti, nulli o corrotti (`NaN`).
* **Distribuzione delle Etichette:** Viene estratta e stampata la frequenza delle cinque classi emotive (da 0 a 4) per avere una prima panoramica quantitativa sul bilanciamento del dataset grezzo prima delle fasi di filtraggio.

In [11]:

with h5py.File(DB_PATH, 'r') as hf:
    # 1. Verifica le "chiavi" (devono essere 'X' e 'y')
    print("Dataset contenuti:", list(hf.keys()))
    
    # 2. Verifica le dimensioni (Shape)
    print("Shape di X (Segnali):", hf['X'].shape) # Deve essere (14691, 62, 1500)
    print("Shape di y (Metadati):", hf['y'].shape) # Deve essere (14691, 3)
    
    # 3. Verifica i tipi di dato
    print("Dtype di X:", hf['X'].dtype) # Deve essere float32

Dataset contenuti: ['X', 'y']
Shape di X (Segnali): (14691, 62, 1500)
Shape di y (Metadati): (14691, 4)
Dtype di X: float32


In [12]:
with h5py.File(DB_PATH, 'r') as hf:
    # Controlliamo un campione casuale o l'intero dataset se la CPU regge
    # Facciamolo a blocchi per sicurezza
    has_nan = False
    for i in range(0, hf['X'].shape[0], 1000):
        block = hf['X'][i:i+1000]
        if np.isnan(block).any():
            has_nan = True
            break
    print(f"Presenza di NaN nel dataset: {has_nan}")

Presenza di NaN nel dataset: False


In [13]:
with h5py.File(DB_PATH, 'r') as hf:
    y_emozioni = hf['y'][:, 2] # Prendi solo la colonna delle etichette
    conteggio = pd.Series(y_emozioni).value_counts().sort_index()
    print("\nDistribuzione Etichette (0:Disgust, 1:Fear, 2:Sad, 3:Neutral, 4:Happy):")
    print(conteggio)


Distribuzione Etichette (0:Disgust, 1:Fear, 2:Sad, 3:Neutral, 4:Happy):
0    2469
1    3006
2    3831
3    2955
4    2430
Name: count, dtype: int64


# 2. Filtraggio per Confidenza Emotiva

Nella seconda fase del processo viene eseguito il preprocessing e la pulizia del dataset precedentemente generato. Poiché l'obiettivo è addestrare il modello su segnali in cui l'emozione target è stata effettivamente provata in modo netto, abbiamo introdotto un filtro di confidenza basato sull'autovalutazione degli utenti. 

Nello specifico, sono stati scartati tutti i campioni EEG associati a uno score inferiore a 3 (su una scala da 1 a 5), mantenendo esclusivamente le finestre temporali relative a risposte emotive di intensità medio-alta. 

Per ragioni di efficienza computazionale e per non saturare la memoria RAM del sistema (dato il notevole peso dei tensori), l'estrazione dei dati puliti avviene in due step. Inizialmente, il sistema legge in memoria solo la matrice dei metadati per individuare e mappare gli indici esatti dei segmenti "validi" (con score $\ge 3$). Successivamente, viene creato un nuovo file HDF5 di destinazione. Le matrici EEG (le cui dimensioni restano invariate a $62 \times 1500$) e i relativi metadati vengono estratti dal dataset grezzo e trasferiti nel dataset pulito procedendo a blocchi iterativi (chunking).

Il risultato di questa fase è un dataset filtrato e ottimizzato, contenente esclusivamente i campioni ad alta affidabilità emotiva, pronto per le eventuali e successive operazioni di pulizia del segnale (es. rimozione degli artefatti, normalizzazione) o direttamente per il training della rete neurale.

In [14]:
# Nomi dei file
input_h5 = DB_PATH    # Dataset originale (creato nel blocco precedente)
output_h5 = DB_PATH_FILTERED  # Nuovo file che conterrà solo i dati pre-processati

print(f"Apertura del dataset originale: {input_h5}")

with h5py.File(input_h5, 'r') as hf_in:
    # 1. Carichiamo solo i metadati (y) in RAM, poiché sono leggeri (matrice N x 4)
    y_original = hf_in['y'][:]
    
    # Ricordiamo la struttura: [Patient_ID, Session_ID, Label, Score]
    # Lo Score è l'ultima colonna (indice 3)
    scores = y_original[:, 3]
    
    # 2. Troviamo le righe (indici) dei campioni che vogliamo mantenere (Score >= 3)
    # np.where restituisce una tupla, [0] estrae l'array di indici
    valid_indices = np.where(scores >= 3)[0]
    
    # Calcolo delle statistiche
    n_total = len(scores)
    n_valid = len(valid_indices)
    n_dropped = n_total - n_valid
    
    print(f"Campioni totali originari: {n_total}")
    print(f"Campioni da mantenere (Score >= 3): {n_valid}")
    print(f"Campioni da scartare (Score < 3): {n_dropped}")
    
    if n_valid == 0:
        print("Nessun campione supera il filtro. Operazione interrotta.")
    else:
        print(f"\nCreazione del nuovo dataset filtrato: {output_h5}")
        
        with h5py.File(output_h5, 'w') as hf_out:
            # 3. Creiamo i nuovi dataset contenitori con le dimensioni esatte (n_valid)
            X_out = hf_out.create_dataset('X', shape=(n_valid, 62, 1500), dtype='float32', chunks=True)
            y_out = hf_out.create_dataset('y', shape=(n_valid, 4), dtype='int8')
            
            # Salviamo subito i metadati aggiornati (già presenti in RAM)
            y_out[:] = y_original[valid_indices]
            
            # 4. Trasferiamo i dati EEG (X)
            # Li copiamo a "blocchi" (chunk) per non sovraccaricare la memoria del PC.
            chunk_size = 500  # Copia 500 campioni temporali alla volta
            
            print("Trasferimento dei tensori EEG in corso...")
            for i in range(0, n_valid, chunk_size):
                # Calcolo limite finale del blocco corrente
                end_idx = min(i + chunk_size, n_valid)
                
                # Estraiamo la lista degli indici originali che corrispondono a questo blocco
                batch_indices = valid_indices[i:end_idx]
                
                # h5py permette di usare una lista di indici ordinati per estrarre dati dal disco
                X_batch = hf_in['X'][batch_indices.tolist()]
                
                # Scriviamo il blocco sul nuovo disco
                X_out[i:end_idx] = X_batch
                
                print(f"  -> Progresso: copiati {end_idx}/{n_valid} campioni...")

print("\nProcesso completato! Dataset pulito è pronto in:", output_h5)

Apertura del dataset originale: ../data/processed/eeg_dataset.h5
Campioni totali originari: 14691
Campioni da mantenere (Score >= 3): 13392
Campioni da scartare (Score < 3): 1299

Creazione del nuovo dataset filtrato: ../data/processed/eeg_dataset_filtered.h5
Trasferimento dei tensori EEG in corso...
  -> Progresso: copiati 500/13392 campioni...
  -> Progresso: copiati 1000/13392 campioni...
  -> Progresso: copiati 1500/13392 campioni...
  -> Progresso: copiati 2000/13392 campioni...
  -> Progresso: copiati 2500/13392 campioni...
  -> Progresso: copiati 3000/13392 campioni...
  -> Progresso: copiati 3500/13392 campioni...
  -> Progresso: copiati 4000/13392 campioni...
  -> Progresso: copiati 4500/13392 campioni...
  -> Progresso: copiati 5000/13392 campioni...
  -> Progresso: copiati 5500/13392 campioni...
  -> Progresso: copiati 6000/13392 campioni...
  -> Progresso: copiati 6500/13392 campioni...
  -> Progresso: copiati 7000/13392 campioni...
  -> Progresso: copiati 7500/13392 campio

# 3. Estrazione delle Feature (Frequency Domain) DE + DASM + RASM:



Pipeline ottimizzata seguendo:

- **Protocollo ufficiale SEED-V** (https://bcmi.sjtu.edu.cn/home/seed/seed-v.html):
  bandpass 1–75 Hz, downsampling a 200 Hz, **Differential Entropy** (DE) per banda
  assumendo segnali Gaussiani: $DE = \tfrac{1}{2}\log(2\pi e\,\sigma^2)$.
- **Survey "Emotions Recognition Using EEG Signals"** (Alarcão & Fonseca, IEEE T-AC 2019,
  `docs/survey_eegemotion.pdf`): i lobi **frontale e parietale** sono i più informativi per
  l'emozione, le bande **Alpha/Beta/Gamma** le più discriminative, e le **feature di asimmetria**
  (DASM, RASM) sono biomarcatori chiave per la valenza emotiva: l'asimmetria frontale alpha
  codifica l'asse approach/withdrawal, l'asimmetria temporale gamma codifica valenza+arousal.

**Feature calcolate per ogni epoca da 1.5 s:**

| Famiglia | Definizione                                                                | # feature      |
|----------|----------------------------------------------------------------------------|---------------:|
| DE       | $\tfrac{1}{2}\log(2\pi e\,\sigma^2_\text{band})$ per ogni canale × banda  | 62 × 5 = **310** |
| DASM     | $DE_\text{left} - DE_\text{right}$ per ogni coppia simmetrica × banda      | 27 × 5 = **135** |
| RASM     | $DE_\text{left} / DE_\text{right}$ per ogni coppia simmetrica × banda      | 27 × 5 = **135** |
| **Tot.** |                                                                            | **580**          |

più 4 colonne meta (`Patient_ID`, `Session_ID`, `Label`, `Score`).

**Preprocessing applicato a ogni epoca prima del calcolo feature:**

1. Conversione in µV (× 1e6) per avere DE su scala leggibile.
2. Bandpass Butterworth zero-phase **1–50 Hz** (la nostra Gamma top è 50 Hz, quindi non serve
   il 75 Hz di SEED-V).
3. Notch **50 Hz** (frequenza di rete EU).
4. **Common Average Reference**.

Il DE viene poi calcolato per ogni banda **filtrando di nuovo** il segnale con un Butterworth
specifico di quella banda, e prendendo $\tfrac{1}{2}\log(2\pi e\,\text{Var}(x_\text{band}))$.
Questa formulazione è quella canonica usata negli script SEED.

Output → `../data/processed/eeg_dataset_FE.csv`.

In [16]:
# ==========================================
# CONFIGURAZIONE
# ==========================================
SFREQ           = 1000              # Hz nel file HDF5
LOWCUT, HIGHCUT = 1.0, 50.0         # Bandpass globale (la Gamma top è 50 Hz)
NOTCH_FREQ      = 50.0              # Frequenza di rete (Italia)
NOTCH_Q         = 30.0

# Bande SEED-V standard
BANDS = {
    'Delta': (1, 4),
    'Theta': (4, 8),
    'Alpha': (8, 14),
    'Beta':  (14, 31),
    'Gamma': (31, 50),
}

RAW_DIR    = '../data/raw'
USELESS_CH = ['M1', 'M2', 'VEO', 'HEO']
CSV_OUTPUT = '../data/processed/eeg_dataset_FE.csv'

# .cnt di riferimento per recuperare i nomi dei canali nello stesso ordine usato
# dal generatore HDF5 (tutti i .cnt SEED-V condividono lo stesso layout 62-ch).
REF_CNT = os.path.join(RAW_DIR, '1_1_20180804.cnt')

### Topologia dei Canali e Asimmetria Emisferica

Per calcolare le metriche DASM e RASM, è necessario confrontare l'attività dei due emisferi cerebrali. Invece di mappare manualmente le coppie, utilizziamo la convenzione del sistema internazionale 10-20 (e 10-10): i canali con suffisso **dispari** si trovano nell'emisfero sinistro (es. F3), mentre i corrispondenti con suffisso **pari** si trovano nell'emisfero destro (es. F4). 

La funzione seguente scansiona i nomi dei 62 canali e accoppia automaticamente i sensori simmetrici, individuando 27 coppie essenziali per l'estrazione delle asimmetrie spaziali.

In [ ]:
# ==========================================
# CANALI E COPPIE SIMMETRICHE
# ==========================================
ref_raw = mne.io.read_raw_cnt(REF_CNT, preload=False, verbose=False)
ref_raw.drop_channels(USELESS_CH, on_missing='ignore')
CH_NAMES = list(ref_raw.ch_names)
del ref_raw
assert len(CH_NAMES) == 62, f"Mi aspetto 62 canali, trovati {len(CH_NAMES)}"
print(f"Canali ({len(CH_NAMES)}): {CH_NAMES}")


def find_symmetric_pairs(ch_names):
    """Trova coppie L–R basandosi sulla convenzione 10-20 / 10-10:
    suffisso DISPARI = sinistra, suffisso PARI = destra (es. F3 ↔ F4).
    Ritorna [(idx_L, idx_R, name_L, name_R), ...]."""
    name_to_idx = {n.upper(): i for i, n in enumerate(ch_names)}
    pairs = []
    for i, name in enumerate(ch_names):
        u = name.upper()
        if not u or not u[-1].isdigit():
            continue
        last = int(u[-1])
        if last % 2 == 1:                       # numero dispari = canale sinistro
            right = u[:-1] + str(last + 1)
            if right in name_to_idx:
                ridx = name_to_idx[right]
                pairs.append((i, ridx, name, ch_names[ridx]))
    return pairs


PAIRS = find_symmetric_pairs(CH_NAMES)
print(f"\nCoppie simmetriche trovate: {len(PAIRS)}")
for li, ri, lname, rname in PAIRS:
    print(f"  {lname:5s}  ↔  {rname}")

Canali (62): ['FP1', 'FPZ', 'FP2', 'AF3', 'AF4', 'F7', 'F5', 'F3', 'F1', 'FZ', 'F2', 'F4', 'F6', 'F8', 'FT7', 'FC5', 'FC3', 'FC1', 'FCZ', 'FC2', 'FC4', 'FC6', 'FT8', 'T7', 'C5', 'C3', 'C1', 'CZ', 'C2', 'C4', 'C6', 'T8', 'TP7', 'CP5', 'CP3', 'CP1', 'CPZ', 'CP2', 'CP4', 'CP6', 'TP8', 'P7', 'P5', 'P3', 'P1', 'PZ', 'P2', 'P4', 'P6', 'P8', 'PO7', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'PO8', 'CB1', 'O1', 'OZ', 'O2', 'CB2']

Coppie simmetriche trovate: 27
  FP1    ↔  FP2
  AF3    ↔  AF4
  F7     ↔  F8
  F5     ↔  F6
  F3     ↔  F4
  F1     ↔  F2
  FT7    ↔  FT8
  FC5    ↔  FC6
  FC3    ↔  FC4
  FC1    ↔  FC2
  T7     ↔  T8
  C5     ↔  C6
  C3     ↔  C4
  C1     ↔  C2
  TP7    ↔  TP8
  CP5    ↔  CP6
  CP3    ↔  CP4
  CP1    ↔  CP2
  P7     ↔  P8
  P5     ↔  P6
  P3     ↔  P4
  P1     ↔  P2
  PO7    ↔  PO8
  PO5    ↔  PO6
  PO3    ↔  PO4
  CB1    ↔  CB2
  O1     ↔  O2


### Filtraggio e Stabilità Numerica (Design delle Funzioni)

In questa sezione definiamo le funzioni matematiche per la trasformazione del segnale:
* **Preprocessing:** Viene applicato un filtro passa-banda (1-50 Hz) e un filtro Notch (50 Hz) bidirezionale (`sosfiltfilt`) per azzerare lo sfasamento.
* **Differential Entropy (DE):** Viene calcolata assumendo che il segnale EEG filtrato per una specifica banda segua una distribuzione Gaussiana.
* **Stabilità Numerica:** Nel calcolo della *Rational Asymmetry* (RASM), che prevede un rapporto tra emisferi ($DE_{left} / DE_{right}$), è stato introdotto un limite inferiore al denominatore (`1e-8`) unito alla conservazione del segno. Questo previene errori di calcolo critici (divisione per zero o overflow) mantenendo l'integrità matematica della feature.

In [ ]:
# ==========================================
# DESIGN FILTRI
# ==========================================
# Bandpass globale 1–50 Hz e notch 50 Hz applicati a ogni epoca prima del DE.
sos_band         = butter(4, [LOWCUT, HIGHCUT], btype='band', fs=SFREQ, output='sos')
b_notch, a_notch = iirnotch(w0=NOTCH_FREQ, Q=NOTCH_Q, fs=SFREQ)

# Filtri per-banda (uno per ogni banda di interesse) usati per calcolare il DE
# direttamente dalla varianza del segnale band-pass filtrato.
SOS_PER_BAND = {
    name: butter(4, [lo, hi], btype='band', fs=SFREQ, output='sos')
    for name, (lo, hi) in BANDS.items()
}


def preprocess_segment(seg):
    """Bandpass 1–50 Hz + notch 50 Hz + Common Average Reference. Input in µV."""
    seg = sosfiltfilt(sos_band, seg, axis=-1)
    seg = filtfilt(b_notch, a_notch, seg, axis=-1)
    seg = seg - seg.mean(axis=0, keepdims=True)
    return seg


def differential_entropy(seg, sos):
    """DE = 0.5 * log(2*pi*e*sigma^2) per ogni canale, calcolato sulla varianza
    del segnale FILTRATO nella banda specifica (assunzione gaussiana SEED)."""
    filt = sosfiltfilt(sos, seg, axis=-1)
    var  = np.var(filt, axis=-1)
    return 0.5 * np.log(2.0 * np.pi * np.e * (var + 1e-12))


def compute_features(seg, pairs):
    """Calcola DE (62 × 5) + DASM (27 × 5) + RASM (27 × 5) per un'epoca."""
    de = np.zeros((seg.shape[0], len(BANDS)), dtype=np.float32)
    for b_idx, b_name in enumerate(BANDS):
        de[:, b_idx] = differential_entropy(seg, SOS_PER_BAND[b_name])

    left_idx  = [p[0] for p in pairs]
    right_idx = [p[1] for p in pairs]
    de_l = de[left_idx]
    de_r = de[right_idx]

    dasm = de_l - de_r
    # RASM con denominatore protetto: preserva il segno e mette un floor in modulo
    safe_de_r = np.sign(de_r) * np.maximum(np.abs(de_r), 1e-8)
    rasm = de_l / safe_de_r
    return de, dasm, rasm

### Esecuzione della Pipeline e Batch Processing

L'applicazione di filtri IIR (Butterworth) e il calcolo della varianza su decine di migliaia di matrici sono operazioni computazionalmente molto onerose. 

Per ottimizzare le performance e prevenire crash per esaurimento della memoria RAM (Out-Of-Memory), l'estrazione delle feature non avviene su tutto il dataset contemporaneamente. I dati vengono caricati dal disco rigido (file HDF5 pre-filtrato) in **chunk da 500 segmenti alla volta**. Ogni blocco viene convertito in microvolt (µV), processato, e i risultati (580 feature + 4 metadati per epoca) vengono accodati per generare il DataFrame finale, che viene infine esportato in formato CSV.

In [19]:
# ==========================================
# PIPELINE: HDF5 → preprocessing → DE+DASM+RASM → CSV
# ==========================================
de_col_names   = [f'DE_{ch}_{band}'        for ch in CH_NAMES                for band in BANDS]
dasm_col_names = [f'DASM_{l}_{r}_{band}'    for (_, _, l, r) in PAIRS         for band in BANDS]
rasm_col_names = [f'RASM_{l}_{r}_{band}'    for (_, _, l, r) in PAIRS         for band in BANDS]
META_COLS      = ['Patient_ID', 'Session_ID', 'Label', 'Score']
ALL_COLS       = META_COLS + de_col_names + dasm_col_names + rasm_col_names

print(
    f"Colonne totali: {len(ALL_COLS)}  "
    f"({len(META_COLS)} meta + {len(de_col_names)} DE + "
    f"{len(dasm_col_names)} DASM + {len(rasm_col_names)} RASM)"
)

rows = []
print("\nInizio estrazione feature...")
with h5py.File(DB_PATH_FILTERED, 'r') as hf:
    X      = hf['X']
    y_meta = hf['y']
    total  = X.shape[0]

    batch = 500
    for i in range(0, total, batch):
        end_idx = min(i + batch, total)
        X_batch = X[i:end_idx]
        y_batch = y_meta[i:end_idx]

        for j in range(X_batch.shape[0]):
            # Conversione in µV (i .cnt MNE forniscono dati in V) + preprocessing
            seg = preprocess_segment(X_batch[j].astype(np.float64) * 1e6)

            de, dasm, rasm = compute_features(seg, PAIRS)

            rows.append([
                int(y_batch[j, 0]), int(y_batch[j, 1]),
                int(y_batch[j, 2]), int(y_batch[j, 3]),
                *de.flatten(), *dasm.flatten(), *rasm.flatten(),
            ])

        if end_idx % 1000 == 0 or end_idx == total:
            print(f"  Processati {end_idx}/{total} segmenti...")

df = pd.DataFrame(rows, columns=ALL_COLS)
df.to_csv(CSV_OUTPUT, index=False)
print(f"\nDataset salvato: {CSV_OUTPUT}")
print(f"Shape: {df.shape}")
del rows, df; gc.collect()

Colonne totali: 584  (4 meta + 310 DE + 135 DASM + 135 RASM)

Inizio estrazione feature v5...
  Processati 1000/13392 segmenti...
  Processati 2000/13392 segmenti...
  Processati 3000/13392 segmenti...
  Processati 4000/13392 segmenti...
  Processati 5000/13392 segmenti...
  Processati 6000/13392 segmenti...
  Processati 7000/13392 segmenti...
  Processati 8000/13392 segmenti...
  Processati 9000/13392 segmenti...
  Processati 10000/13392 segmenti...
  Processati 11000/13392 segmenti...
  Processati 12000/13392 segmenti...
  Processati 13000/13392 segmenti...
  Processati 13392/13392 segmenti...

Dataset v5 salvato: ../data/processed/eeg_dataset_FE.csv
Shape: (13392, 584)


0

### Verifica e Validazione delle Feature (Sanity Check Finale)

In questa cella eseguiamo tre validazioni rapide:
* **Ispezione Visiva:** Controlliamo la *shape* del file e stampiamo le prime righe per assicurarci che le colonne dei metadati e delle feature siano allineate correttamente.
* **Distribuzione delle Classi (Label):** Verifichiamo il conteggio finale dei campioni per ogni emozione. Questo è essenziale per capire il bilanciamento del dataset prima di passare all'addestramento della rete.
* **Validazione Statistica (Differential Entropy):** Analizziamo i limiti (min, max, media) della metrica DE. Poiché i segnali EEG convertiti in microvolt (µV) presentano tipicamente una varianza di pochi µV, ci aspettiamo che la formula logaritmica della DE produca valori indicativamente ristretti nel range **[-2, +6]**. Valori fortemente al di fuori di questo intervallo sarebbero la "spia" di un errore nel preprocessing (es. mancata conversione della scala o rumore anomalo non filtrato).

In [20]:
# ==========================================
# VERIFICA RAPIDA
# ==========================================
head = pd.read_csv(CSV_OUTPUT, nrows=5)
print(f"Shape (head): {head.shape}")
print("\nPrime 9 colonne (4 meta + 5 DE):")
print(head.iloc[:, :9])

labels = pd.read_csv(CSV_OUTPUT, usecols=['Label'])['Label']
print("\nDistribuzione Label:")
print(labels.value_counts().sort_index())

# Sanity check sui valori DE: per segnali EEG in µV con σ dell'ordine di pochi µV,
# DE dovrebbe stare grosso modo nel range [-2, +6].
de_only = pd.read_csv(CSV_OUTPUT, usecols=lambda c: c.startswith('DE_'))
print(f"\nDE stats: min={de_only.values.min():.2f}  max={de_only.values.max():.2f}  "
      f"mean={de_only.values.mean():.2f}  std={de_only.values.std():.2f}")

Shape (head): (5, 584)

Prime 9 colonne (4 meta + 5 DE):
   Patient_ID  Session_ID  Label  Score  DE_FP1_Delta  DE_FP1_Theta  \
0           1           1      1      4      3.059196      2.849478   
1           1           1      1      4      3.220361      2.844981   
2           1           1      1      4      3.353453      2.591909   
3           1           1      1      4      5.011252      4.206511   
4           1           1      1      4      3.564834      3.154073   

   DE_FP1_Alpha  DE_FP1_Beta  DE_FP1_Gamma  
0      2.557815     3.251494      3.537333  
1      2.674771     3.427274      3.364406  
2      2.623242     3.489843      3.045092  
3      3.328722     3.488515      3.236767  
4      2.918969     3.276929      3.461145  

Distribuzione Label:
Label
0    1900
1    2711
2    3555
3    2955
4    2271
Name: count, dtype: int64

DE stats: min=-0.00  max=7.16  mean=1.93  std=0.57


# 4.Data Splitting

In questa fase finale, il dataset contenente le 580 feature estratte viene suddiviso in tre set indipendenti. Questa operazione è cruciale per valutare correttamente la capacità di generalizzazione dei modelli di Machine Learning ed evitare fenomeni di *overfitting*.

La procedura segue questi passaggi chiave:

1.  **Shuffle e Stratificazione:** Prima dello split, i dati vengono rimescolati casualmente. Viene utilizzata la **stratificazione** sulla colonna `Label`: questo garantisce che la proporzione delle 5 classi emotive (Disgust, Fear, Sad, Neutral, Happy) rimanga identica in tutti e tre i set, evitando sbilanciamenti che potrebbero influenzare l'addestramento.
2.  **Ripartizione delle Quote (70/15/15):**
    * **Train Set (70%):** Utilizzato per l'addestramento vero e proprio dei modelli.
    * **Validation Set (15%):** Utilizzato per il fine-tuning degli iperparametri e per monitorare le performance durante l'addestramento.
    * **Test Set (15%):** Riservato esclusivamente alla valutazione finale delle prestazioni su dati mai visti dal modello.
3.  **Esportazione:** Come indicato nelle note tecniche, la normalizzazione è un passaggio critico per i segnali EEG. I dati vengono salvati in file CSV separati nella directory `../data/splitted/` per garantire la riproducibilità degli esperimenti, risparmiare tempo computazionale nelle sessioni successive e mantenere il codice di addestramento pulito e modulare.

Importazione File

In [21]:
csv_filename = '../data/processed/eeg_dataset_FE.csv'

print("Caricamento dataset in memoria...")
df = pd.read_csv(csv_filename)


Caricamento dataset in memoria...


Divisione dataset

In [ ]:
# --- 1.1 SHUFFLE DEL DATAFRAME ---
# Rimescoliamo tutto il DataFrame (frac=1 significa il 100% delle righe) 
# e resettiamo l'indice in modo che parta da zero in modo pulito.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# La nostra label di riferimento per la stratificazione è la colonna 'Label'
y_labels = df['Label'].values

# --- 1.2 ETICHETTE E SPLIT ---
# Split 1: Separiamo il Test set (15%) dal resto (85% Temp)
df_temp, df_test, y_temp, y_test = train_test_split(
    df, y_labels, test_size=0.15, random_state=42, stratify=y_labels, shuffle=True
)

# Split 2: Dal Temp (85%), estraiamo il Validation set.
# Per avere il 15% del totale, calcoliamo la frazione: 0.15 / 0.85
val_fraction = 0.15 / 0.85
df_train, df_val = train_test_split(
    df_temp, test_size=val_fraction, random_state=42, stratify=y_temp, shuffle=True
)

print(f"Dataset pronto. \n Split: Train {df_train.shape[0]}, Val {df_val.shape[0]}, Test {df_test.shape[0]}")

Dataset pronto. Split: Train 9374, Val 2009, Test 2009


Salvataggio su disco

In [23]:

# Definisci i percorsi di salvataggio
save_dir = '../data/splitted/'
os.makedirs(save_dir, exist_ok=True) # Crea la cartella se non esiste

train_path = os.path.join(save_dir, 'eeg_train.csv')
val_path   = os.path.join(save_dir, 'eeg_val.csv')
test_path  = os.path.join(save_dir, 'eeg_test.csv')

def save_split_to_csv(filename, dataframe):
    """Salva il dataframe in un file CSV."""
    print(f"Salvataggio in corso: {filename}...")
    # index=False è fondamentale per non salvare la colonna con i vecchi indici di riga pandas
    dataframe.to_csv(filename, index=False)
    print(f"Completato. Shape: {dataframe.shape}")

# Esecuzione del salvataggio
save_split_to_csv(train_path, df_train)
save_split_to_csv(val_path, df_val)
save_split_to_csv(test_path, df_test)

print("\nTutti gli split sono stati esportati con successo.")

Salvataggio in corso: ../data/splitted/eeg_train.csv...
Completato. Shape: (9374, 584)
Salvataggio in corso: ../data/splitted/eeg_val.csv...
Completato. Shape: (2009, 584)
Salvataggio in corso: ../data/splitted/eeg_test.csv...
Completato. Shape: (2009, 584)

Tutti gli split sono stati esportati con successo.
